## Code Generator

The requirement: use a Frontier model to generate high performance Code Converter

In [ ]:
# imports

import os
import io
import sys
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
import re

In [ ]:
# Setup Gemini Client
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

In [ ]:
# Connect to client libraries

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
GEMINI_MODEL = "gemini-3.1-flash-lite"

In [ ]:
# Prompts
system_prompt = """
Your task is to convert code from one programming language to another.
Respond only with the requested code. Do not provide any explanation other than occasional comments.
The converted code must be highly optimized, idiomatic, and produce identical output to the original.
"""

def user_prompt_for(source_code, input_lang, output_lang):
    return f"""
Port this {input_lang} code to {output_lang} with the most algorithmically efficient implementation that produces identical output.
Focus on optimal data structures, cache locality, and standard library features appropriate for {output_lang}.

Respond only with {output_lang} code.
Code to port:

```{input_lang.lower()}
{source_code}
```
"""

In [ ]:
def port(source_code, input_lang, output_lang):
    if input_lang == output_lang:
        return source_code
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(source_code, input_lang, output_lang)}
        ]
    response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages, reasoning_effort="high")
    reply = response.choices[0].message.content.strip()
    # Safely extract ONLY the code block content, ignoring conversational padding
    match = re.search(r'```[\w+#]*\n(.*?)\n```', reply, re.DOTALL)
    if match:
        return match.group(1).strip()
        
    return reply.strip()

In [ ]:
pi = """
def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

result = calculate(200_000_000, 4, 1) * 4
"""

In [ ]:
# Validate the function by running a python code calculating its time take for generating the output
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

print(run_python(pi))

## Compiling C++ and executing

Run the C++ code in online compiler to test it out that way. 
For example here: https://www.programiz.com/cpp-programming/online-compiler/

Result: 3.141592656089 \
Execution Time: 2.097847 seconds

In [ ]:
# Checking the Port Function is working correctly or not for the python code created earlier
port("Python", "C++", pi)

In [ ]:
f"{132.583159/2.097847:.0f}X speedup"

In [ ]:
FULL_PAGE_CSS = """
body, html {
    margin: 0;
    padding: 0;
}
.gradio-container {
    max-width: 100% !important;
    padding: 15px 20px !important;
}
/* Safely constrain the code areas to 65% of the screen height */
.code-row {
    height: 65vh !important; 
    margin-bottom: 15px !important;
}
/* Kill the redundant outer scrollbars */
.code-container {
    height: 100% !important;
    overflow: hidden !important; 
}
.code-container > div {
    height: 100% !important;
    overflow: hidden !important; 
}
/* Let CodeMirror handle its own internal scrolling natively */
.cm-editor {
    height: 100% !important;
}
/* Button styling */
.convert-btn {
    background: linear-gradient(135deg, #10a37f, #0d8a6a) !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: bold !important;
    padding: 12px !important;
    box-shadow: 0 4px 14px 0 rgba(16, 163, 127, 0.3) !important;
    transition: all 0.2s ease-in-out !important;
    margin-top: 10px !important; 
}
.convert-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(16, 163, 127, 0.4) !important;
}
"""

In [ ]:
# Keep only natively supported languages for gr.Code()
LANG_MAP = {
    "Python": "python", 
    "C++": "cpp", 
    "JavaScript": "javascript", 
    "TypeScript": "typescript"
}

def update_syntax(lang):
    # Fallback to None (plain text) if the language isn't supported
    safe_lang = LANG_MAP.get(lang, None)
    return gr.Code(language=safe_lang)

In [ ]:
# Pre-compute the list once to optimize speed
ALL_LANGS = list(LANG_MAP.keys())

def update_output_choices(selected_input, current_output):
    # Fast list comprehension to filter choices
    valid_choices = [lang for lang in ALL_LANGS if lang != selected_input]
    # Keep current value if valid, else default to the first available choice
    new_value = current_output if current_output in valid_choices else valid_choices[0]
    return gr.update(choices=valid_choices, value=new_value)

def update_input_choices(selected_output, current_input):
    valid_choices = [lang for lang in ALL_LANGS if lang != selected_output]
    new_value = current_input if current_input in valid_choices else valid_choices[0]
    return gr.update(choices=valid_choices, value=new_value)

In [ ]:
# Create a Gradio UI that has handful of supporting gr.Code() languages and use that only as of now for our laguage converter
# Note: Our LLM can convert to any language but we are sticking to fewer options here

with gr.Blocks(css=FULL_PAGE_CSS, theme=gr.themes.Soft(), title="Universal Code Converter") as ui:
    gr.Markdown("## 🌐 Universal Code Converter (Powered by Gemini)")
    with gr.Row():
        with gr.Column(scale=1):
            input_lang = gr.Dropdown(
                choices=ALL_LANGS, 
                value="Python", 
                label="Input Language"
            )
        with gr.Column(scale=1):
            output_lang = gr.Dropdown(
                choices=[lang for lang in ALL_LANGS if lang != "Python"], 
                value="C++", 
                label="Output Language"
            )

    with gr.Row(elem_classes=["code-row"]):
        with gr.Column(scale=1, elem_classes=["code-container"]):
            source_code = gr.Code(
                label="Source Code",
                value="",
                language="python"
            )
        with gr.Column(scale=1, elem_classes=["code-container"]):
            converted_code = gr.Code(
                label="Converted Code",
                value="",
                language="cpp"
            )

    convert_btn = gr.Button("⚡ Convert Code", elem_classes=["convert-btn"])

    input_lang.change(fn=update_output_choices, inputs=[input_lang, output_lang], outputs=[output_lang])
    output_lang.change(fn=update_input_choices, inputs=[output_lang, input_lang], outputs=[input_lang])

    # Update syntax highlighting live when dropdowns change
    input_lang.change(fn=update_syntax, inputs=[input_lang], outputs=[source_code])
    output_lang.change(fn=update_syntax, inputs=[output_lang], outputs=[converted_code])

    # Convert code
    convert_btn.click(
        fn=port, 
        inputs=[source_code, input_lang, output_lang], 
        outputs=[converted_code]
    )

ui.launch(inbrowser=True)